# DSSAT Converter - Run Main Function

This notebook runs the standard DSSAT converter. It generates DSSAT input files and executes each simulation independently, without DSSAT sequence mode.

## 1. Locate the Repository and Import Modules

In [6]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
repo_root = next(
    (
        path
        for path in candidates
        if (path / "tests" / "data").exists() and (path / "src" / "modfilegen").exists()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(f"Could not locate the ModFileGen repository from cwd={cwd}.")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from modfilegen import GlobalVariables
from modfilegen.Converter.DssatConverter.dssatconverter import fetch_data_from_sqlite, main

print(f"Kernel cwd: {cwd}")
print(f"Repo root: {repo_root}")

Kernel cwd: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/notebooks
Repo root: /mnt/d/Mes Donnees/TCMP/github/ModFileGen


## 2. Configure Paths

In [7]:
data_dir = repo_root / "tests" / "data"
output_dir = repo_root / "tests" / "output_dssat"
temp_dir = output_dir / "temp"

master_input_db = data_dir / "MasterInput.db"
models_dict_db = data_dir / "ModelsDictionaryArise.db"
cultivars_folder = data_dir / "cultivars" / "dssat"

output_dir.mkdir(parents=True, exist_ok=True)
temp_dir.mkdir(parents=True, exist_ok=True)

n_threads = 1
n_parts = 1
# dt=0 keeps generated DSSAT folders for inspection. Set dt=1 to clean intermediate files after successful runs.
dt = 0

print(f"Master Input DB exists: {master_input_db.exists()} -> {master_input_db}")
print(f"Models Dict DB exists: {models_dict_db.exists()} -> {models_dict_db}")
print(f"Cultivars folder exists: {cultivars_folder.exists()} -> {cultivars_folder}")
print(f"Output directory: {output_dir}")
print(f"Temp directory: {temp_dir}")

Master Input DB exists: True -> /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/MasterInput.db
Models Dict DB exists: True -> /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/ModelsDictionaryArise.db
Cultivars folder exists: True -> /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/cultivars/dssat
Output directory: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat
Temp directory: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat/temp


## 3. Preview Simulations

In [8]:
rows = fetch_data_from_sqlite(str(master_input_db))
print(f"Total simulations: {len(rows)}")

for index, row in enumerate(rows[:10], start=1):
    print(
        f"{index}. {row['idsim']} | "
        f"site={row.get('idPoint')} start={row.get('StartYear')}-{row.get('StartDay')} "
        f"end={row.get('EndYear')}-{row.get('EndDay')}"
    )

Total simulations: 9
1. 5.925_6.025_2000_Mgt1M0_135_2 | site=5.925_6.025 start=2000-105 end=2000-335
2. 5.925_6.025_2001_Mgt1M0_135_2 | site=5.925_6.025 start=2001-105 end=2001-335
3. 5.925_6.025_2002_Mgt1M0_135_2 | site=5.925_6.025 start=2002-105 end=2002-335
4. 5.925_6.025_2000_Mgt1M0_150_2 | site=5.925_6.025 start=2000-120 end=2000-350
5. 5.925_6.025_2001_Mgt1M0_150_2 | site=5.925_6.025 start=2001-120 end=2001-350
6. 5.925_6.025_2002_Mgt1M0_150_2 | site=5.925_6.025 start=2002-120 end=2002-350
7. 5.925_6.025_2000_Mgt1M0_155_2 | site=5.925_6.025 start=2000-125 end=2000-355
8. 5.925_6.025_2001_Mgt1M0_155_2 | site=5.925_6.025 start=2001-125 end=2001-355
9. 5.925_6.025_2002_Mgt1M0_155_2 | site=5.925_6.025 start=2002-125 end=2002-355


## 4. Set Global Variables

In [9]:
GlobalVariables["dbMasterInput"] = str(master_input_db)
GlobalVariables["dbModelsDictionary"] = str(models_dict_db)
GlobalVariables["directorypath"] = str(output_dir)
GlobalVariables["pltfolder"] = str(cultivars_folder)
GlobalVariables["nthreads"] = n_threads
GlobalVariables["dt"] = dt
GlobalVariables["parts"] = n_parts
GlobalVariables["tempDir"] = str(temp_dir)

print("GlobalVariables configured:")
for key, value in GlobalVariables.items():
    print(f"  {key}: {value}")

GlobalVariables configured:
  storeNumMinSimu: 0
  storeNumMaxSimu: 0
  storeKeyDataN: 0
  dbMasterInput: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/MasterInput.db
  dbModelsDictionary: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/ModelsDictionaryArise.db
  dbCelsius: 
  dt: 0
  ori_MI: 
  parts: 1
  tempDir: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat/temp
  package: 
  directorypath: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat
  pltfolder: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/cultivars/dssat
  nthreads: 1


## 5. Run the DSSAT Converter

In [10]:
print("Starting DSSAT conversion...")
print("=" * 60)

try:
    main()
    print("\n" + "=" * 60)
    print("DSSAT conversion completed successfully.")
except Exception as error:
    print("\n" + "=" * 60)
    print(f"Error during DSSAT conversion: {error}")
    import traceback

    traceback.print_exc()
    raise

Starting DSSAT conversion...
Indexes created successfully!
📊 Total simulations to process: 9
Processing 1 chunks...
Iteration 0
Iteration 1
Iteration 2
Iteration 3
Iteration 4
Iteration 5
Iteration 6
Iteration 7
Iteration 8
✅ Chunk 1/1: 9 rows written
✅ Results saved to /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat/4fca04fc-2b54-43cc-b8bb-b3af51fef1a3_dssat.csv
DSSAT total time, 9.075688600540161

DSSAT conversion completed successfully.


## 6. Inspect Generated Results

In [ ]:
result_files = sorted(output_dir.glob("*_dssat.csv"), key=lambda path: path.stat().st_mtime)
print(f"DSSAT result files: {len(result_files)}")

if result_files:
    latest_result = result_files[-1]
    print(f"Latest result: {latest_result}")
    import pandas as pd

    display(pd.read_csv(latest_result).head())
else:
    print("No DSSAT result CSV found yet.")